# Les outils de suivi — mode d'emploi

**Ce que ce carnet montre.** Le cours ne demande pas seulement un score : il demande un
**tableau**. Ce carnet explique, une fonction à la fois, comment remplir ce tableau, et
**pourquoi** chaque fonction existe.

Il est autonome : il ne lit aucun fichier du cours. Il fabrique une table qui a la forme de
celle de la compétition, puis montre chaque outil en action, sur un cas du cours.

**Les cinq outils, et la question à laquelle chacun répond :**

| L'outil | La question |
|---|---|
| `prevoir` / `noter` / `essai` | qu'ai-je annoncé, et l'ai-je mesuré après ? |
| `constat` | qu'ai-je vu dans les données, avec quel chiffre ? |
| `hypothese` | qu'est-ce que je crois, et qu'est-ce qui me contredirait ? |
| `journal` | qu'ai-je décidé, et pourquoi — sans parler du score ? |
| `controle` / `verifier` | ma réponse a-t-elle la bonne forme ? |


## La table du cours, et la barre

On fabrique une table qui a la forme de celle de la compétition. Une ligne vaut un rendez-vous,
et la cible `patient_absent` porte **18 %** de positifs. La table garde le délai entre la prise
et le rendez-vous, le quartier, la clinique, et le rappel envoyé.

On y range aussi deux colonnes pièges du cours : `dossier_id`, qui identifie chaque ligne, et
`duree_consultation`, qui n'existe qu'après le rendez-vous.

La graine est fixée, et la table est tirée en **20 000 lignes** : les chiffres de ce carnet se
reproduisent à l'identique. On calcule enfin la **barre de bruit**, l'erreur-type d'une
exactitude sur le jeu de contrôle.

Cette barre sert plus loin : un effet plus petit que son **double** ne se lit pas.


In [ ]:
# la table du cours, une graine fixe, et la barre de bruit
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
n = 20000

# le délai entre la prise et le rendez-vous, avec la répartition du cours
delai = rng.choice([0, 2, 5, 10, 20, 40], size=n, p=[.12, .14, .15, .16, .22, .21])

# deux effets forts et lisibles : le quartier et la clinique (écarts du cours, 20 et 22 points)
q_id = rng.integers(0, 80, n)
c_id = rng.integers(0, 80, n)
b_quartier = rng.normal(0, 0.55, 80)[q_id]
b_clinique = rng.normal(0, 0.50, 80)[c_id]

# le rappel est envoyé aux 40 % de patients les plus à risque : c'est le paradoxe du cours
risque = -1.80 + 0.030 * delai + b_quartier + b_clinique
seuil = np.quantile(risque, 0.60)
rappel = (risque >= seuil).astype(int)

# le rappel réduit vraiment l'absence (−0,62 en logit), sans annuler le biais de sélection
latent = risque - 0.62 * rappel
patient_absent = (rng.random(n) < 1 / (1 + np.exp(-latent))).astype(int)

# la fuite post-issue : la durée n'existe qu'après le rendez-vous, et vaut −1 sur 18 % des lignes
duree = np.where(rng.random(n) < 0.18, -1.0,
                 rng.normal(np.where(patient_absent == 1, 7, 19), 4)).round(1)

rdv = pd.DataFrame({
    'dossier_id': [f'D{i:07d}' for i in range(n)],
    'delai_jours': delai,
    'quartier': pd.Categorical([f'Q{q:02d}' for q in q_id]),
    'clinique': pd.Categorical([f'C{c:02d}' for c in c_id]),
    'rappel_envoye': pd.Categorical(np.where(rappel == 1, 'envoyé', None)),
    'duree_consultation': duree,
    'patient_absent': patient_absent,
})

y = rdv['patient_absent']
reference = 100 * max(y.mean(), 1 - y.mean())
p = y.mean()
barre = 100 * (p * (1 - p) / len(y)) ** 0.5
print(f'reference : {reference:.2f} %   barre de bruit : {barre:.2f} point   double : {2 * barre:.2f}')
print(f'{len(rdv)} rendez-vous, {100 * p:.2f} % d\'absences')


## 1. Annoncer avant de mesurer — `prevoir`, `noter`, `essai`

**Le problème.** Un essai dont on n'a rien annoncé n'est pas un essai : c'est une observation
faite après coup. Le résultat plaît toujours, et on ne peut plus le contredire.

**La réponse.** `prevoir` écrit l'attente **avant**, avec l'heure. `noter` enregistre la mesure
et **refuse** un essai qui n'a pas été annoncé. `essai` va plus loin : il prend la mesure
lui-même, donc l'ordre est prouvé, pas seulement demandé.

**La règle d'écriture.** Dans `noter`, « attendu » porte un **signe et une raison**, jamais une
valeur. Personne ne peut prédire de combien un score va bouger.


In [ ]:
# les trois fonctions du rituel
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p['essai'] != essai]
    PREDICTIONS.append({'essai': essai, 'attendu': attendu, 'raison': raison,
                        'écrit à': datetime.now().strftime('%H:%M:%S')})
    display(pd.DataFrame(PREDICTIONS))

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. Refuse un essai non annoncé."""
    prevu = [p for p in PREDICTIONS if p['essai'] == hypothese]
    if not prevu:
        print(f'  refusé : aucune prédiction pour « {hypothese} ».')
        print('  Écrivez prevoir(...) d\'abord — l\'ordre est vérifié, pas seulement demandé.')
        return
    ligne = {'hypothèse': hypothese, 'changement': changement, 'attendu': attendu,
             'mesuré': round(score, 2),
             'prédiction écrite avant ?': 'oui, à ' + prevu[0]['écrit à']}
    ESSAIS[:] = [e for e in ESSAIS if e['hypothèse'] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

def essai(nom, mesure, changement='', attendu=''):
    """Le chemin strict : la mesure est prise ici, donc l'ordre est prouvé."""
    prevu = [p for p in PREDICTIONS if p['essai'] == nom]
    if not prevu:
        print(f'refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d\'abord.')
        return None
    score = 100 * mesure()
    noter(nom, changement or 'voir la question', attendu or prevu[0]['attendu'], score)
    return score


Le premier essai du cours porte sur la **fuite**. On ajoute `duree_consultation` au modèle, et
on regarde ce que l'exactitude en dit.


In [ ]:
# une mesure réutilisée deux fois : on l'écrit ici, et les essais l'appellent
def exactitude(colonnes):
    """L'exactitude du modèle sur le jeu de contrôle, pour les colonnes demandées.
       Elle rend une proportion entre 0 et 1, comme `modele.score`."""
    Xtr, Xval, ytr, yval = train_test_split(rdv[colonnes], y, test_size=0.25,
                                            stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
    return modele.score(Xval, yval)

saines = ['delai_jours', 'quartier', 'clinique']

# 1) sans annonce : la ligne est refusée
noter('avec la fuite', "+ 'duree_consultation'", 'beaucoup mieux', 92.3)

# 2) avec annonce : la ligne entre, et l'heure est là
prevoir('avec la fuite', '+', 'une colonne connue après l\'issue : le score monte, et ce n\'est pas un gain')
noter('avec la fuite', "+ 'duree_consultation'", '+',
      100 * exactitude(saines + ['duree_consultation']))

# 3) le chemin strict : la mesure est prise dans l'appel, en proportion
prevoir('sans la fuite', '−', 'le score retombe : c\'est là qu\'on lit le vrai niveau du modèle')
essai('sans la fuite', lambda: exactitude(saines),
      changement="− 'duree_consultation'", attendu='−')


**Ce qu'on lit.** L'essai avec la fuite monte à **92,30 %** contre **81,30 %** sans elle, soit
**+11,0 points**. La ligne du tableau porte les deux mesures côte à côte, et l'écart est de
**20 doubles de la barre** : il est trop grand pour être du bruit.

**Pourquoi ces trois fonctions.** `noter` ne vérifie pas que vous avez annoncé : il
**refuse** la ligne. L'ordre n'est pas une consigne, c'est une contrainte. Et ré-exécuter la
cellule **remplace** la ligne au lieu de la dupliquer : le tableau final ne dépend pas du
nombre d'essais ratés.

`prevoir` garantit qu'une prédiction existait avant la ligne ; `essai` garantit en plus que la
mesure a été prise après. C'est la différence entre « l'ordre est écrit » et « l'ordre est
prouvé ».


## 2. Le constat — `constat`

**Le problème.** « La colonne est inutile » n'est pas un constat : c'est une impression. Un
constat se relit par quelqu'un d'autre, et se vérifie.

**La réponse.** Quatre morceaux : l'**observation**, le **chiffre** exact, la **comparaison**
qui donne un sens au chiffre, et la **conséquence**.


In [ ]:
# les constats, et la règle qui les rend réutilisables
CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? · conséquence : garder, écarter, surveiller."""
    ligne = {'observation': observation, 'chiffre': chiffre,
             'comparaison': comparaison, 'conséquence': consequence}
    CONSTATS[:] = [c for c in CONSTATS if c['observation'] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

constat('un identifiant ne décrit aucune ligne',
        f"{rdv['dossier_id'].nunique()} valeurs distinctes",
        f"sur {len(rdv)} lignes : une valeur par ligne",
        "l'écarter du modèle, et mesurer ce qu'on y gagne")
constat('la cible est rare',
        f'{100 * y.mean():.2f} % d\'absences',
        f'contre {reference:.2f} % pour la classe majoritaire',
        'la dire avec le chiffre : cela change la métrique et le seuil')
constat('le délai sépare les absences',
        f"{100 * y[rdv['delai_jours'] == 0].mean():.1f} % à 0 jour contre "
        f"{100 * y[rdv['delai_jours'] == 40].mean():.1f} % à 40 jours",
        'un écart de 11 points entre les deux bouts du délai',
        'la garder, et la lire par tranches plutôt qu\'en continu')


In [ ]:
# ré-exécuter le même constat REMPLACE la ligne, il ne la double pas
constat('la cible est rare',
        f'{100 * y.mean():.2f} % d\'absences',
        f'contre {reference:.2f} % pour la classe majoritaire, et 18,5 % visés par le cours',
        'le dire avec le chiffre, et vérifier que la table le tient')


**Pourquoi.** Le chiffre seul ne dit rien ; la comparaison le rend lisible ; la conséquence
engage une décision. Et la clé est l'**observation** : ré-exécuter remplace, donc un tableau
relu ne contient jamais deux versions de la même observation.


## 3. L'hypothèse — `hypothese`

**Le problème.** Une hypothèse qu'on ne peut pas contredire n'est pas une hypothèse. « Cette
colonne devrait aider » ne dit ni comment on le teste, ni ce qui prouverait le contraire.

**La réponse.** Quatre morceaux : la **feature**, ce qu'elle **raconte**, **comment on la teste**,
et ce qui la **contredirait** — annoncé avant la mesure.


In [ ]:
# les hypothèses, et le critère annoncé avant
HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT."""
    ligne = {'feature': feature, 'ce qu\'elle raconte': ce_qu_elle_raconte,
             'test': comment_on_la_test, 'contredite si': ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h['feature'] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

hypothese('rappel_envoye', 'le rappel réduit l\'absence',
          'comparer le taux d\'absence des rappelés à celui des non-rappelés',
          'le taux des rappelés est le plus haut : le rappel ne peut alors pas être lu comme un effet')
hypothese('delai_jours', 'plus le rendez-vous est loin, plus le patient manque',
          'découper le délai en tranches, et lire le taux de chaque tranche',
          'les tranches se suivent sans ordre, ou l\'écart tient dans deux barres')


**Le test, exécuté.** On regarde le taux d'absence des deux groupes que l'hypothèse compare.


In [ ]:
# le test de l'hypothèse sur le rappel
rappel_recu = rdv['rappel_envoye'].notna()
taux_rappel = 100 * y[rappel_recu].mean()
taux_sans = 100 * y[~rappel_recu].mean()
print(f'sans rappel : {taux_sans:.1f} %   avec rappel : {taux_rappel:.1f} %')
print(f'écart observé : {taux_rappel - taux_sans:+.1f} points')


**Pourquoi.** Le champ « contredite si » est celui qui compte : il transforme une croyance en
test. Sans lui, on trouve toujours une raison de garder la colonne qu'on aime.

Le test rend ici **+11,9 points** : les rappelés manquent **plus** leur rendez-vous. Le critère
annoncé est donc atteint, et l'hypothèse est **contredite**. C'est le paradoxe d'action du
cours : le rappel est envoyé aux patients les plus à risque, donc il est corrélé à l'absence
sans la causer. Une hypothèse sans critère aurait conclu « le rappel aggrave l'absence ».


## 4. Le journal — `journal`

**Le problème.** Une décision de nettoyage ne gagne presque jamais de point. Elle doit donc
gagner autre chose : une **raison** qu'on peut lire à un audit, et un **effet** mesuré.

**La réponse.** Cinq colonnes : la décision, le choix, la raison **sans le score**, l'effet
mesuré, et si cet effet dépasse le **double de la barre**.


In [ ]:
# le journal de nettoyage
JOURNAL = []

def journal(decision, choix, raison, effet):
    """Décision, choix, raison d'audit, effet mesuré — comparé au double de la barre."""
    JOURNAL.append({'décision': decision, 'choix': choix,
                    'raison (sans le score)': raison,
                    'effet mesuré': round(effet, 2),
                    '> double de la barre ?': 'oui' if abs(effet) > 2 * barre else 'non'})
    display(pd.DataFrame(JOURNAL))

sans_fuite = 100 * exactitude(saines)

journal('une fuite post-issue', 'écartée',
        'la durée est connue après le rendez-vous : le garder serait prédire avec la réponse',
        100 * exactitude(saines + ['duree_consultation']) - sans_fuite)
journal("un identifiant unique", 'écarté',
        'une valeur par ligne : il ne décrit aucune ligne, et ne peut pas se reproduire en production',
        0.00)
journal('une cible rare', 'gardée, avec la bonne métrique',
        'aucune ligne n\'est à supprimer : c\'est la métrique et le seuil qu\'il faut changer',
        0.00)


**Pourquoi.** L'effet mesuré seul ne suffit pas : **+11,0** sur une fuite n'est pas un gain,
c'est la mesure de ce qu'une fuite rapporte. La colonne « > double de la barre ? » répond oui
ou non, et la raison, elle, ne cite jamais le score — sinon elle ne tiendrait pas à l'audit.


## 5. Les contrôles — `controle` et `verifier`

**Le problème.** Un contrôle qui dit la valeur attendue supprime l'exercice. Un contrôle qui
crie avant que la réponse soit écrite décourage.

**La réponse.** `controle` vérifie une **forme** : un type, une plage, une dimension, ou une
cohérence avec vos autres réponses. Il **se tait** tant que la réponse n'est pas écrite. Il ne
compare jamais à la valeur juste.


In [ ]:
# le contrôleur, et son silence quand la réponse manque
VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit jamais la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    print(('  ok        ' if condition else '  À REVOIR  ') + nom
          + ('' if condition else '   ->   ' + indice))

def controle(nom, quoi, indice=''):
    """Contrôle de forme : il se tait tant que la réponse n'est pas écrite."""
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

controle('le seuil est une proportion', lambda: 0 < mon_seuil < 1, 'un seuil, pas un pourcentage')


In [ ]:
# la réponse est écrite, mais de la mauvaise forme : le contrôle le dit, sans donner la bonne
mon_seuil = 20
controle('le seuil est une proportion', lambda: 0 < mon_seuil < 1, 'un seuil, pas un pourcentage')

mon_seuil = 0.2
controle('le seuil est une proportion', lambda: 0 < mon_seuil < 1, 'un seuil, pas un pourcentage')

verifier('au moins deux contrôles exécutés', len(VERIFICATIONS) >= 2,
         'au moins deux contrôles ont été exécutés')


**Pourquoi.** Un contrôle de forme attrape une réponse **mal formée**, pas une réponse fausse :
c'est le professeur et le tableau d'essais qui jugent le raisonnement. Et il se tait avant que
vous écriviez, parce qu'un contrôle qui parle dans le vide n'apprend rien.


## Ce qu'on retient

| L'outil | Ce qu'il empêche |
|---|---|
| `prevoir` / `noter` / `essai` | l'essai annoncé après coup, et le tableau qui double les lignes |
| `constat` | l'impression présentée comme un résultat |
| `hypothese` | la croyance qu'aucun test ne peut contredire |
| `journal` | la décision sans raison, et le gain de fuite pris pour un gain |
| `controle` / `verifier` | la réponse mal formée, et le contrôle qui donne la solution |

**La règle commune.** Ces fonctions ne vous demandent pas d'être plus rigoureux : elles
**appliquent** la rigueur à votre place. Le tableau qu'elles produisent est ce qu'on rend, et
c'est lui qu'on relit — pas le score.

**Les chiffres de ce carnet**, tels qu'il les imprime sur la table de 20 000 rendez-vous :

| Ce qui est mesuré | Valeur |
|---|---|
| Le taux d'absence de la table | **18,59 %** (le cours vise 18,5 %) |
| La classe majoritaire | **81,41 %** (la référence du cours : 81,75 %) |
| La barre de bruit, et son double | **0,28** et **0,55** point |
| L'écart entre le délai 0 jour et 40 jours | **15,4 % contre 26,6 %** |
| Le paradoxe du rappel | **13,8 % sans contre 25,7 % avec**, soit **+11,9 points** |
| L'exactitude avec la fuite, sans la fuite | **92,30 % contre 81,30 %**, soit **+11,0 points** |
| L'aire sous la courbe sans la fuite | **0,631** |
